In [ ]:
from collections.abc import Callable, Iterable
import json
from pathlib import Path

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData

from jazz_graph.data.graph_builder import CreateTensors
from jazz_graph.data.fetch import fetch_recording_traits, fetch_discogs_to_recording_id
from jazz_graph.model.model import LinkPredictionModel, UnsupervisedJazzModel
from jazz_graph.recommendation.recommender import Recommender, LookupRecordings, PredictLinkRecommender, RandomWalkRecommender
from jazz_graph.metrics.ranking import map_at_k
from jazz_graph.recommendation.experiment import BSideExperiment
from jazz_graph.training.logging import ExperimentLogger, load_embeddings, load_model, find_most_recent_run

checkpoint_path = None
# checkpoint_path = '/workspace/experiments/2026-03-26_02-01-03_gnn_simCLR_graph_parquet'  # This was good, but I think we need to retrain on new data.
if checkpoint_path is None:
    checkpoint_path = str(find_most_recent_run('/workspace/experiments', 'simCLR'))

with open(Path(checkpoint_path) / 'config.json', 'r') as f:
    run_config = json.loads(f.read())
    nodes_data_path = run_config['data_config'].get('dataset')

experiment_config = {
    'gnn': checkpoint_path,
    'nodes_data': nodes_data_path,
}


In [ ]:
run_config

In [ ]:
def lookup_from_dataset(data: HeteroData) -> LookupRecordings:
    # But why? Answer: the graph data may include pruning that is not part of the
    # parquet datasets or recording traits.
    rec_ids = data['performance'].x[:, 1].numpy()
    ids = np.arange(rec_ids.size)
    df = pd.DataFrame({'recording_ids': rec_ids, 'ids': ids}).set_index('recording_ids')
    return LookupRecordings(df)


In [ ]:
class UnsupervisedModelAdapter(torch.nn.Module):
    def __init__(self, model: UnsupervisedJazzModel):
        super().__init__()
        self.model = model

    def __call__(self, x_dict, edge_index_dict, batch):
        return self.model(batch)
        return self.model.encode(batch)

In [ ]:
def iter_dirs(root: str):
    import os
    for base_dir in os.listdir(root):
        print(base_dir)
        break

iter_dirs('/workspace/experiments')

In [ ]:
## Inductive Graph Recommender
from sklearn import model_selection
from jazz_graph.data.graph_builder import make_jazz_data
from jazz_graph.model.model import JazzModel
from jazz_graph.recommendation.recommender import InferenceRecommender


graph_data: HeteroData = make_jazz_data(CreateTensors(nodes_data_path))
model_state = load_model(checkpoint_path)
model_state = model_state.get('model_state_dict', model_state)
# model_state = model_state.get('model_state_dict', model_state)
model = UnsupervisedJazzModel.from_config(run_config)
model.load_state_dict(model_state)
model = UnsupervisedModelAdapter(model)
recommender = InferenceRecommender(model, graph_data)
lookup = recommender.lookup_recordings
recording_traits = fetch_recording_traits(use_proto=nodes_data_path.endswith('proto')).set_index('recording_id')
experimenter = BSideExperiment(recommender, recording_traits)

In [ ]:

album_experiments = [
    ('Miles Davis', 'Kind of Blue'),
    ('Miles Davis', 'Sketches of Spain'),
    ('Art Blakey & The Jazz Messengers', 'Mosaic'),
    ('Charles Mingus', "Mingus Ah Um"),  # lots of songs, should have some.
    ('The Dave Brubeck Quartet', "Time Out"),
    ('Ornette Coleman', 'The Shape of Jazz to Come')  # very unusual music--should probably be easy.
]

In [ ]:
recording_traits.dtypes

In [ ]:
experiment = album_experiments[-2]
print(experiment)
result = experimenter.b_side_precision(*experiment)
result
recording_traits.loc[result['recommended_ids']]

In [ ]:
results = experimenter.b_side_experiment(experiment_config, album_experiments)
pd.DataFrame.from_records(results).drop(columns=['recommended_ids'])

## Spotify Data

In [ ]:
import os
import json
from jazz_graph.recommendation.playlist import SpotifyListens
from jazz_graph.recommendation.recommender import ArtistWeightedRecommender
from jazz_graph.clean.data_normalization import normalize_title
import random

from jazz_graph.recommendation.experiment import SpotifyExperiement, RandomAlbumSplit


In [ ]:
spotify = 'local_data'
spotify_sample = '../local_data/my_spotify_data/spotify_extended_streaming_history/Streaming_History_Audio_2024-2025_4.json'
with open(spotify_sample, 'r') as f:
    spotify_data = json.loads(f.read())

def spotify_details(record: dict):
    details = [
        'ts', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name',
        'reason_start', 'reason_end', 'shuffle', 'skipped'
    ]
    return {key: record.get(key) for key in details}

spotify_data[-100:-90]
spotify_data[-10:]
ten_recent = [spotify_details(record) for record in spotify_data[-20:-10]]

In [ ]:

# seen = set()
# i = 0
# for rec in spotify_data:
#     title = rec['master_metadata_track_name']
#     norm_title = normalize_title(title)
#     if norm_title in seen:
#         continue
#     seen.add(norm_title)
#     if 'digital' in title.lower():
#         i += 1
#         print(norm_title)
#         print(title)
#     # print(norm_title)
# print(i)

In [ ]:
def test_rand_walk_recommender(graph_data):
    kob = recording_traits[recording_traits.album == "Kind of Blue"]
    baseline_recommender = RandomWalkRecommender(graph_data, 42)
    recommendations, scores, mask = baseline_recommender.get_recommendations(kob.index.to_list())
    ex_shape = recommendations.shape
    assert scores.shape == ex_shape
    assert mask.dtype == bool
    assert mask.shape == ex_shape
    return recommendations


# test_recs = test_rand_walk_recommender(graph_data)
# recording_traits.loc[test_recs]

In [ ]:
baseline_recommender = RandomWalkRecommender(graph_data, 42)
simple_artist_recommender = ArtistWeightedRecommender(recording_traits)


In [ ]:
# fields = 'master_metadata_track_name', 'master_metadata_album_album_name', 'master_metadata_album_artist_name'
# for recording in spotify_data:
#     for field in fields:
#         if recording.get(field) is None:
#             print(recording)

In [ ]:
experiment = SpotifyExperiement(recording_traits, spotify_data)

In [ ]:
experiment.run_experiment(baseline_recommender, {'model': "RandomWalkRecommender", "seed": 42})

In [ ]:
experiment.run_experiment(simple_artist_recommender, {'model': "ArtistWeightedRecommender"})

In [ ]:
experiment.run_experiment(recommender, experiment_config)


In [ ]:
def load_rec_run(path):
    path = Path(path)
    with open(path / 'metrics.jsonl') as f:
        return json.loads(f.read())

results = load_rec_run(str(find_most_recent_run('/workspace/experiments', 'spotify_recommendations')))
# results = load_rec_run('/workspace/experiments/2026-03-24_16-58-12_spotify_recommendations')
print(results['samples'].keys())
recording_traits.loc
samples = results['samples']
assert np.setdiff1d(samples['true_positive_in_familiar_recommendation'], samples['true_positive_in_familiar_recommendation']).size == 0

In [ ]:
tp_familiar = results['samples']['true_positive_in_familiar_recommendation']
recording_traits.loc[tp_familiar].value_counts('artist')

In [ ]:
results['samples'].keys()

In [ ]:
false_negatives = results['samples']['false_negative_in_novel_sample']
true_positives = results['samples']['true_positive_in_novel_recommendation']
recording_traits.loc[false_negatives].sort_values('release_date')
recording_traits.loc[true_positives][:50]
novel_recs = results['samples']['top_1302_novel_recommendations']
recording_traits.loc[novel_recs].head(50)
new_novel_recs = np.setdiff1d(novel_recs, true_positives)
recording_traits.loc[new_novel_recs][400:450]

## Explore misses in Spotify Matching

In [ ]:
import re
recording_traits[(recording_traits.artist.str.contains('Coltrane')) & recording_traits.album.str.contains('live', flags=re.IGNORECASE)].head(50)
None

In [ ]:
find_misses = SpotifyListens(recording_traits)
misses = [e[0] for e in find_misses.get_spotify_misses(spotify_data)]
print(len(misses), len(spotify_data))

In [ ]:
[key for key in find_misses.lookup if key[2].startswith('wayne s') and 'ad' in key[1]]

In [ ]:
recording_traits[recording_traits['album'].str.contains("Adam")]

In [ ]:
misses[0]

In [ ]:
unique = set()
clean_misses = []
for record in misses:
    keys = 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name'

    details = spotify_details(record)
    key = tuple(details[e] for e in keys)
    if key in unique:
        continue
    details.pop('ts')

    e = {key: details[other_key] for key, other_key in zip(['song', 'artist', 'album', ], keys)}
    clean_misses.append(e)
    unique.add(key)


In [ ]:
clean_misses

In [ ]:
[e for e in clean_misses if 'Gloria' in e['song']]

In [ ]:


def match_recording_traits(recording_traits, artist=None, album=None):
    def starts_with_lower(key, match):
        value = recording_traits[key]
        return value.str.contains(match, flags=re.IGNORECASE)

    if artist and album:
        return recording_traits[starts_with_lower('artist', artist) & starts_with_lower('album', album)]
    elif artist:
        return recording_traits[starts_with_lower('artist', artist)]
    elif album:
        return recording_traits[starts_with_lower('album', album)]


In [ ]:
re.search('^(Sunday )?at the', 'At the Village', re.IGNORECASE)

In [ ]:

result = match_recording_traits(recording_traits, 'Wayne Shorter', album='Adam')
#print(len(result))
result.sort_values('release_date').sort_values('title')#.loc[1662631]

In [ ]:
sunday = [spotify_details(v) for v in misses if v['master_metadata_album_album_name'].startswith('Sunday')]
normalize_title(sunday[0]['master_metadata_track_name']) == normalize_title(result.sort_values('release_date').loc[1662631].title)


In [ ]:
(sunday[0]['master_metadata_track_name'], result.sort_values('release_date').loc[1662631].title)

In [ ]:

result = match_recording_traits(recording_traits, album="N")
print(len(result))
result

In [ ]:
recording_traits.loc[experiment.in_samples].query("artist.str.contains('John')")

In [ ]:
recording_traits.loc[experiment.out_samples].query("artist.str.contains('John')")

In [ ]:
novel_recs = recording_traits.loc[recommendations[~mask]][:500]
all_recs = recording_traits.loc[recommendations][:500].assign(rank=np.arange(1, 501))
novel_recs = novel_recs.merge(all_recs['rank'], left_index=True, right_index=True)
repeat_idx = all_recs.index.difference(novel_recs.index)
repeat_recs = all_recs.loc[repeat_idx]
repeat_recs.head(30)

In [ ]:
from jazz_graph.etl.extract_discogs import InMemDiscogs, is_jazz_album

discogs = InMemDiscogs('/workspace/local_data/jazz_releases.jsonl', is_jazz_album)

In [ ]:
mismatches =[
    ('A Love Supreme', "song titles can be weird w/ Pt. I etc."),
    ('The Bill Evans Trio', "THE"),
    ('Equinox', 'Album naima, JC. Missing in discogs.'),
    ("Ascenseur pour l'échafaud", "This was  sound track by Miles in 1957. It's borked.")
    ("Feio", "no discogs match (Bitches Brew track.)")
]

In [ ]:
discogs.tracklist().get(normalize_title('naima'))

In [ ]:
discogs.get_albums_matching_title(normalize_title('a love supreme'))

In [ ]:
import jsonlines
with jsonlines.open(discogs.release_path) as f:
    for release in f:
        if normalize_title(release['title']) == normalize_title('the art of conversation'):
            print(release)

In [ ]:
import psycopg
query = """
    SELECT * FROM recording_first_release
    WHERE
        album ILIKE 'ada%' AND
        artist ILIKE 'Wayne Shorter'
        -- AND title ILIKE 'eq%'
    LIMIT 100;
"""
with psycopg.connect("dbname=musicbrainz_db user=philosofool") as conn:
    query_result = pd.read_sql(query, conn)
query_result

In [ ]:
query_result.album.loc[0]

In [ ]:
for row in query_result.itertuples():
    print(discogs.matching_discog(row))

In [ ]:
from jazz_graph.etl.extract_discogs import MatchDiscogs, InMemDiscogs, is_jazz_album
discogs = MatchDiscogs(InMemDiscogs('/workspace/local_data/jazz_releases.jsonl', is_jazz_album))

In [ ]:
discogs.matching_discog((None, None, "Adam's Apple", "Adam's Apple", "Wayne Shorter"))

In [ ]:
embeddings = torch.tensor([
    [.1, .1],
    [1, 0],
    [.1, .2],
    [.2, .1],
    [.3, .3]
])

dp = embeddings @ embeddings[[1, 3]].T
dp
dp.sum(dim=-1)